# Exploration ingestion — opendataloader-pdf standard vs hybrid

Objectif : comparer l'extraction PDF **standard** (pipeline Java actuel, `backend/app/ingestion/loaders/pdf_loader.py`) et l'extraction **hybride** (`opendataloader-pdf[hybrid]`, backend `docling-fast`) sur quelques documents de la loi de finance tunisienne, pour décider de la stratégie de chunking à retenir avant de modifier le code d'ingestion.

Kernel : `fiscalmind` (venv du projet, `backend/.venv`).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]  # backend/notebooks -> backend -> repo root
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("repo root:", REPO_ROOT)

from backend.app.ingestion.loaders.pdf_loader import extract_pdf
from backend.app.ingestion.chunking import chunk_document, split_into_articles

import opendataloader_pdf

SAMPLE_PDFS = [
    REPO_ROOT / "data" / "processed" / "sample_test" / "src" / "Loi-de-Finances-2006.pdf",  # gros fichier (6.9 Mo) -> probablement scanné
    REPO_ROOT / "data" / "raw" / "pdfs" / "cdpf-2025.pdf",  # code permanent, probablement texte natif
]
for p in SAMPLE_PDFS:
    print(p, "exists:", p.exists(), "size_kb:", round(p.stat().st_size / 1024, 1) if p.exists() else None)

## 1. Extraction standard (baseline, sans hybride)

C'est ce que fait actuellement `extract_pdf()` en production (pipeline Java opendataloader-pdf, pas de OCR/mise en page avancée).

In [ ]:
standard_results = {}
for pdf_path in SAMPLE_PDFS:
    result = extract_pdf(pdf_path)
    standard_results[pdf_path.name] = result
    print(f"--- {pdf_path.name} ---")
    print("pages:", result.page_count, "| chars texte:", len(result.text), "| chars markdown:", len(result.markdown))
    print("warnings:", result.warnings)
    print(result.text[:400].replace("\n", " | "))
    print()

## 2. Démarrage du backend hybride (docling-fast)

Lance `opendataloader-pdf-hybrid` en sous-processus sur le port 5002 et attend qu'il soit prêt. Premier lancement possiblement lent (téléchargement des modèles docling/easyocr).

In [ ]:
import socket
import subprocess
import sys as _sys
import time

HYBRID_PORT = 5002
HYBRID_URL = f"http://127.0.0.1:{HYBRID_PORT}"


def port_open(host: str, port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(1)
        return sock.connect_ex((host, port)) == 0


hybrid_proc = None
if port_open("127.0.0.1", HYBRID_PORT):
    print("Un serveur écoute déjà sur le port", HYBRID_PORT, "- on le réutilise.")
else:
    venv_bin = Path(_sys.executable).parent
    hybrid_cmd = [str(venv_bin / "opendataloader-pdf-hybrid"), "--port", str(HYBRID_PORT)]
    print("Lancement:", " ".join(hybrid_cmd))
    hybrid_proc = subprocess.Popen(
        hybrid_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=str(REPO_ROOT),
    )

    deadline = time.time() + 300  # premiers chargements de modèles peuvent être lents
    while time.time() < deadline:
        if port_open("127.0.0.1", HYBRID_PORT):
            print("Serveur hybride prêt après", round(300 - (deadline - time.time()), 1), "s")
            break
        if hybrid_proc.poll() is not None:
            print("Le serveur s'est arrêté prématurément. Sortie :")
            print(hybrid_proc.stdout.read())
            break
        time.sleep(2)
    else:
        print("Timeout en attendant le serveur hybride.")

## 3. Extraction hybride (`hybrid=docling-fast`)

In [ ]:
import tempfile


def extract_hybrid(pdf_path: Path) -> dict:
    work_dir = Path(tempfile.mkdtemp(prefix="odl_hybrid_"))
    opendataloader_pdf.convert(
        input_path=str(pdf_path),
        output_dir=str(work_dir),
        format="markdown,json",
        hybrid="docling-fast",
        hybrid_url=HYBRID_URL,
        hybrid_fallback=True,
        quiet=True,
    )
    markdown_path = work_dir / f"{pdf_path.stem}.md"
    markdown = markdown_path.read_text(encoding="utf-8", errors="ignore") if markdown_path.exists() else ""
    return {"markdown": markdown, "work_dir": work_dir}


hybrid_results = {}
for pdf_path in SAMPLE_PDFS:
    print(f"--- {pdf_path.name} (hybride) ---")
    try:
        res = extract_hybrid(pdf_path)
        hybrid_results[pdf_path.name] = res
        print("chars markdown:", len(res["markdown"]))
        print(res["markdown"][:400])
    except Exception as exc:
        print("ECHEC:", exc)
    print()

## 4. Comparaison standard vs hybride

Heuristique de qualité structurelle : nombre de marqueurs d'article détectés (`Article N` / `الفصل`) — sert de proxy pour savoir si `split_into_articles()` va bien fonctionner sur chaque sortie.

In [ ]:
from backend.app.ingestion.chunking import _find_article_boundaries

for pdf_path in SAMPLE_PDFS:
    name = pdf_path.name
    std_text = standard_results[name].text
    hyb_markdown = hybrid_results.get(name, {}).get("markdown", "")
    print(f"=== {name} ===")
    print("standard : chars =", len(std_text), "| articles détectés =", len(_find_article_boundaries(std_text)))
    print("hybride  : chars =", len(hyb_markdown), "| articles détectés =", len(_find_article_boundaries(hyb_markdown)))
    print()

## 5. Test de chunking sur les deux extractions

Applique `chunk_document()` (logique actuelle du projet) sur les deux versions de texte pour voir laquelle produit des chunks mieux alignés sur les articles.

In [ ]:
def chunk_stats(source_file: str, text: str) -> dict:
    chunks = chunk_document(source_file, text)
    with_article = sum(1 for c in chunks if c.numero_article is not None)
    avg_len = sum(len(c.text) for c in chunks) / len(chunks) if chunks else 0
    return {"nb_chunks": len(chunks), "avec_numero_article": with_article, "longueur_moyenne": round(avg_len)}


for pdf_path in SAMPLE_PDFS:
    name = pdf_path.name
    std_text = standard_results[name].text
    hyb_markdown = hybrid_results.get(name, {}).get("markdown", "")
    print(f"=== {name} ===")
    print("standard :", chunk_stats(name, std_text))
    if hyb_markdown:
        print("hybride  :", chunk_stats(name, hyb_markdown))
    print()

## 6. Conclusion

_À compléter après revue des résultats ci-dessus : le mode hybride apporte-t-il une amélioration suffisante sur les documents scannés pour justifier son coût (latence, dépendances lourdes) ? Faut-il l'activer uniquement pour les PDFs détectés comme scannés (peu de texte natif) et garder le pipeline standard pour le reste ? La taille de chunk (`DEFAULT_MAX_CHARS=1200`) et la regex de détection d'article restent-elles adaptées aux deux sorties ?_

## 7. Nettoyage

In [ ]:
if hybrid_proc is not None and hybrid_proc.poll() is None:
    hybrid_proc.terminate()
    try:
        hybrid_proc.wait(timeout=10)
    except subprocess.TimeoutExpired:
        hybrid_proc.kill()
    print("Serveur hybride arrêté.")
else:
    print("Rien à arrêter (serveur pré-existant ou jamais démarré).")